In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('cleaned_hr.csv')
df.head()

,Employee_Name,EmpID,MarriedID,MaritalStatusID,GenderID,EmpStatusID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,...,Hire_Month_Name,Hire_Day_Name,Absence_Label,Absence_Id,Absence_Label_id,Col1,Col2,First_Name,Middle_Name,Last_Name
0,Adinolfi Wilson K,10026,0,0,1,1,5,4,0,62506,...,July,Tuesday,Low Absences,0,0,Low,Absences,Wilson,K,Adinolfi
1,Ait Sidi Karthikeyan,10084,1,1,1,5,3,3,0,104437,...,March,Monday,High Absences,2,2,High,Absences,Karthikeyan,NaN,Ait Sidi
2,Akinkuolie Sarah,10196,1,1,0,5,5,3,0,64955,...,July,Tuesday,Low Absences,0,0,Low,Absences,Sarah,NaN,Akinkuolie
3,AlagbeTrina,10088,1,1,0,1,5,3,0,64991,...,January,Monday,High Absences,2,2,High,Absences,Trina,NaN,Alagbe
4,Anderson Carol,10069,0,2,0,5,5,3,0,50825,...,July,Monday,Low Absences,0,0,Low,Absences,Carol,NaN,Anderson


In [3]:
def dateConverter(dataframe, *args):
    for col in args:
        dataframe[col] = pd.to_datetime(dataframe[col], format='mixed')
    print('Datetime converted.')

In [4]:
dateConverter(df, 'DOB', 'DateofHire', 'DateofTermination', 'LastPerformanceReview_Date')

Datetime converted.


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311 entries, 0 to 310
Data columns (total 49 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Employee_Name               311 non-null    object        
 1   EmpID                       311 non-null    int64         
 2   MarriedID                   311 non-null    int64         
 3   MaritalStatusID             311 non-null    int64         
 4   GenderID                    311 non-null    int64         
 5   EmpStatusID                 311 non-null    int64         
 6   DeptID                      311 non-null    int64         
 7   PerfScoreID                 311 non-null    int64         
 8   FromDiversityJobFairID      311 non-null    int64         
 9   Salary                      311 non-null    int64         
 10  Termd                       311 non-null    int64         
 11  PositionID                  311 non-null    int64         

## Extract Year, Month, Day

In [6]:
# df['DateofHire']
'''
Date Number
--------------
dt.year -> year
dt.month -> month
dt.day -> day

Date Name
-------------
dt.day_name() -> Day Name
dt.month_name() -> Month Name
'''

'\nDate Number\n--------------\ndt.year -> year\ndt.month -> month\ndt.day -> day\n\nDate Name\n-------------\ndt.day_name() -> Day Name\ndt.month_name() -> Month Name\n'

In [7]:
df['Hire_Year'] = df['DateofHire'].dt.year
df['Hire_Month'] = df['DateofHire'].dt.month
df['Hire_Day'] = df['DateofHire'].dt.day

In [8]:
df['Hire_Month_Name'] = df['DateofHire'].dt.month_name()
df['Hire_Day_Name'] = df['DateofHire'].dt.day_name()

## Convert Continuous data into Categorical Column

In [9]:
'''
0 - 5 -> Low Absences
6 - 10 -> Medium Absences
10 > -> High Absences
'''

'\n0 - 5 -> Low Absences\n6 - 10 -> Medium Absences\n10 > -> High Absences\n'

In [10]:
def absenceCategory(absence):
    if absence >= 0 and absence <= 5:
        return 'Low Absences'
    elif absence > 5 and absence <= 10:
        return 'Medium Absences'
    else:
        return 'High Absences'

In [11]:
df['Absence_Label'] = df['Absences'].apply(absenceCategory)

## Data Encoding

### Manual Encoding

In [12]:
data = {
    'Low Absences': 0,
    'Medium Absences': 1,
    'High Absences': 2
}

In [13]:
df['Absence_Id'] = df['Absence_Label'].map(data)

In [14]:
def encodeData(dataframe, col_name, *args):
    encode_data = {}
    for idx, val in enumerate(args):
        encode_data[val] = idx

    dataframe[f'{col_name}_id'] = dataframe[col_name].map(encode_data)
    print(f'{col_name} data encoded.')

In [15]:
encodeData(df, 'Absence_Label', 'Low Absences', 'Medium Absences', 'High Absences')

Absence_Label data encoded.


## Split

In [16]:
df['Absence_Label'].unique()
# split(find, number_of_columns=index, expand=True/False(default))

array(['Low Absences', 'High Absences', 'Medium Absences'], dtype=object)

In [17]:
df[['Col1', 'Col2']] = df['Absence_Label'].str.split(' ', n=1, expand=True)

In [18]:
hr = pd.read_csv('Dataset/HR_Dataset Refresh.csv')
hr.drop_duplicates(subset='EmpID', keep='first', inplace=True)

In [19]:
first_last = hr['Employee_Name'].str.split(',', n=1, expand=True)
# first_last.head()

In [20]:
first_middle = first_last[1].str.strip().str.split(' ', n=1, expand=True)

In [21]:
# first_middle.head()

In [29]:
hr['First_Name'] = first_middle[0]
hr['Middle_Name'] = first_middle[1]
hr['Last_Name'] = first_last[0]


In [30]:
df = df.merge(
    hr[['EmpID', 'First_Name', 'Middle_Name', 'Last_Name']],
    on = 'EmpID',
    how='left'
)

In [33]:
df = df.drop(columns=['First_Name_x', 'Middle_Name_x', 'Last_Name_x'])

In [36]:
df.columns = df.columns.str.replace('_y', '')

In [37]:
df.to_csv('cleaned_hr.csv', index=False)

In [38]:
df

,Employee_Name,EmpID,MarriedID,MaritalStatusID,GenderID,EmpStatusID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,...,Hire_Month_Name,Hire_Day_Name,Absence_Label,Absence_Id,Absence_Label_id,Col1,Col2,First_Name,Middle_Name,Last_Name
0,Adinolfi Wilson K,10026,0,0,1,1,5,4,0,62506,...,July,Tuesday,Low Absences,0,0,Low,Absences,Wilson,K,Adinolfi
1,Ait Sidi Karthikeyan,10084,1,1,1,5,3,3,0,104437,...,March,Monday,High Absences,2,2,High,Absences,Karthikeyan,None,Ait Sidi
2,Akinkuolie Sarah,10196,1,1,0,5,5,3,0,64955,...,July,Tuesday,Low Absences,0,0,Low,Absences,Sarah,None,Akinkuolie
3,AlagbeTrina,10088,1,1,0,1,5,3,0,64991,...,January,Monday,High Absences,2,2,High,Absences,Trina,None,Alagbe
4,Anderson Carol,10069,0,2,0,5,5,3,0,50825,...,July,Monday,Low Absences,0,0,Low,Absences,Carol,None,Anderson
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306,Woodson Jason,10135,0,0,1,1,5,3,0,65893,...,July,Monday,High Absences,2,2,High,Absences,Jason,None,Woodson
307,Ybarra Catherine,10301,0,0,0,5,5,1,0,48513,...,September,Tuesday,Low Absences,0,0,Low,Absences,Catherine,None,Ybarra
308,Zamora Jennifer,10010,0,0,0,1,3,4,0,220450,...,April,Saturday,High Absences,2,2,High,Absences,Jennifer,None,Zamora
309,Zhou Julia,10043,0,0,0,1,3,3,0,89292,...,March,Monday,High Absences,2,2,High,Absences,Julia,None,Zhou
